# Notebook 02: Surface curvature mapping

**Paper section:** Results — Surface curvature mapping  
**Paper figures:** Figs. 3 (Gaussian curvature), 4 (Mean curvature)

This notebook computes the local Gaussian and Mean curvature at every vertex of the neural tube lumen mesh.

**Method:** For each vertex, a quadratic surface is fitted to its local neighborhood (adjacent vertices). The principal curvatures k₁ and k₂ are derived from the eigenvalues of the shape operator, and from these:
- Gaussian curvature K = k₁ · k₂  
- Mean curvature H = (k₁ + k₂) / 2

K > 0: synclastic (dome-like) geometry  
K < 0: anticlastic (saddle-shaped) geometry  
K = 0: developable surface (cylindrical or flat)

In [ ]:
from vedo import settings
settings.default_backend = "vtk"

import spatchcocking as sp
import numpy as np
import matplotlib.pyplot as plt

## Load mesh

In [ ]:
mesh = sp.get_mesh("../data/meshes/HH17/HH17_embryo1_lumen.ply")

## Compute curvatures

In [ ]:
# Computes k1, k2, Gauss_curvature (K), Mean_curvature (H)
# and stores them as vertex point arrays on the mesh
mesh = sp.compute_and_save_curvatures(mesh)

K = mesh.pointdata["Gauss_curvature"]
H = mesh.pointdata["Mean_curvature"]
print(f"K: min={K.min():.2e}, max={K.max():.2e}, mean={K.mean():.2e}")
print(f"H: min={H.min():.2e}, max={H.max():.2e}, mean={H.mean():.2e}")

## Visualize on 3D mesh

In [ ]:
from vedo import Plotter

# Gaussian curvature
mesh.pointdata.select("Gauss_curvature")
vmin, vmax = sp.getTightercmap(K, pct=97)
mesh.cmap("PiYG", vmin=vmin, vmax=vmax)

p = Plotter(offscreen=True)
p.show(mesh, axes=0)
p.screenshot("gaussian_curvature_3d.png")
from IPython.display import Image
Image("gaussian_curvature_3d.png")

## Save mesh with curvature data

In [ ]:
sp.save_mesh(mesh, "../data/meshes/HH17/HH17_embryo1_lumen_curvature.ply")

## Distribution of curvature values

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(K * 1e6, bins=100, color="steelblue", edgecolor="none")
axes[0].set_xlabel("Gaussian curvature K [µm⁻²] × 10⁻⁶")
axes[0].set_ylabel("Vertex count")
axes[0].set_title("Gaussian curvature distribution")
axes[0].axvline(0, color="k", linewidth=0.8, linestyle="--")

axes[1].hist(H * 1e3, bins=100, color="darkorange", edgecolor="none")
axes[1].set_xlabel("Mean curvature H [µm⁻¹] × 10⁻³")
axes[1].set_ylabel("Vertex count")
axes[1].set_title("Mean curvature distribution")

plt.tight_layout()
plt.show()